In [ ]:
# StandUp4AI: Verify IoU Evaluation
# Run this cell-by-cell to verify the paper's F1=0.952 @ IoU=0.4 claim
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, numpy as np, pandas as pd, librosa, json
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = os.path.join(BASE, 'audio')
EMNLP_DIR = os.path.join(BASE, 'seq-Standup4AI/dataset/en_uk/emnlp+jahak/all')
SR = 22050

# Quick check
print(f'Audio: {len(os.listdir(AUDIO_DIR))} files')
print(f'EMNLP: {len(os.listdir(EMNLP_DIR))} files')

In [ ]:
def extract_features(audio_path, t0, t1):
    dur = t1 - t0
    if dur < 0.1: return None
    try:
        y, sr = librosa.load(str(audio_path), sr=SR, offset=t0, duration=min(dur, 10.0), mono=True)
    except: return None
    if len(y) < int(0.05 * SR): return None
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
    try: cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    except: cent = np.zeros_like(rms)
    try: bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
    except: bw = np.zeros_like(rms)
    try: roll = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)[0]
    except: roll = np.zeros_like(rms)
    try: flat = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
    except: flat = np.zeros_like(rms)
    feats = [np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.median(rms),
             np.mean(zcr), np.std(zcr), np.max(zcr),
             np.mean(cent), np.std(cent),
             np.mean(bw), np.std(bw),
             np.mean(roll), np.std(roll),
             np.mean(flat), np.std(flat), len(y)/SR]
    if dur >= 0.5:
        try:
            f0, voiced, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            voiced = np.nan_to_num(voiced, nan=0)
            feats += [np.mean(f0), np.std(f0), np.mean(voiced)]
        except: feats += [0, 0, 0]
    else: feats += [0, 0, 0]
    return np.array(feats, dtype=np.float32)

feat = extract_features(f'{AUDIO_DIR}/l2oaxKORheA.m4a', 0.5, 1.5)
print(f'Feature dim: {feat.shape[0]}')

In [ ]:
# Find overlap between audio and EMNLP labels
emlnp_ids = {f.replace('.csv','') for f in os.listdir(EMNLP_DIR) if f.endswith('.csv')}
audio_ids = {f.replace('.m4a','').replace('.mp3','') for f in os.listdir(AUDIO_DIR)
             if f.endswith('.m4a') or f.endswith('.mp3')}
overlap = sorted(emlnp_ids & audio_ids)
print(f'EMNLP: {len(emlnp_ids)}, Audio: {len(audio_ids)}, Overlap: {len(overlap)}')

In [ ]:
# Extract all word segments
X_all, y_all, vids_all = [], [], []
for i, vid in enumerate(overlap):
    audio_path = f'{AUDIO_DIR}/{vid}.m4a'
    label_path = f'{EMNLP_DIR}/{vid}.csv'
    try: df = pd.read_csv(label_path)
    except: continue
    for _, row in df.iterrows():
        ts = eval(str(row['timestamp']))
        t0, t1 = float(ts[0]), float(ts[1])
        feat = extract_features(audio_path, t0, t1)
        if feat is None: continue
        lbl = str(row['label']).strip()
        is_laugh = 1 if lbl in ('B', 'I', 'L') else 0
        X_all.append(feat); y_all.append(is_laugh); vids_all.append(vid)
    if (i+1) % 20 == 0: print(f'  {i+1}/{len(overlap)}')

X = np.array(X_all); y = np.array(y_all); vids = np.array(vids_all)
pos = y.sum()
print(f'Total: {len(X)} segments, {pos} positive ({100*pos/len(y):.1f}%)')

In [ ]:
# Train XGBoost with GroupKFold
n_vids = len(set(vids))
gkf = GroupKFold(n_splits=min(5, n_vids))
models, scalers = [], []
for fold, (tr, te) in enumerate(gkf.split(X, y, vids)):
    sc = StandardScaler()
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
    clf.fit(sc.fit_transform(X[tr]), y[tr])
    models.append(clf); scalers.append(sc)
    print(f'Fold {fold+1}: train={len(tr)}, test={len(te)}')

In [ ]:
# IoU helpers
def span_iou(s1, s2):
    i = max(0.0, min(s1[1], s2[1]) - max(s1[0], s2[0]))
    u = max(s1[1], s2[1]) - min(s1[0], s2[0])
    return i / u if u > 0 else 0.0

def iou_f1(ps, gs, th=0.2):
    if not ps or not gs: return 0.0, 0.0, 0.0
    mp, mg = set(), set()
    for pi, p in enumerate(ps):
        bi, bv = -1, 0.0
        for gi, g in enumerate(gs):
            if gi in mg: continue
            iv = span_iou(p, g)
            if iv >= th and iv > bv: bi, bv = gi, iv
        if bi >= 0: mp.add(pi); mg.add(bi)
    tp = len(mp)
    p = tp/len(ps); r = tp/len(gs)
    return p, r, 2*p*r/(p+r) if (p+r) > 0 else 0.0

p, r, f = iou_f1([(0.0, 1.0)], [(0.5, 1.5)], 0.2)
print(f'IoU test: P={p:.2f} R={r:.2f} F1={f:.2f}')

In [ ]:
# Per-video IoU evaluation
pred_th = 0.5
iou_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
all_results = {th: [] for th in iou_thresholds}

for fold, (tr, te) in enumerate(gkf.split(X, y, vids)):
    Xte = scalers[fold].transform(X[te])
    probs = models[fold].predict_proba(Xte)[:, 1]
    te_vids = vids[te]
    for vid in set(te_vids):
        mask = (te_vids == vid)
        vid_probs = probs[mask]
        df = pd.read_csv(f'{EMNLP_DIR}/{vid}.csv')
        segs = []
        for _, row in df.iterrows():
            ts = eval(str(row['timestamp']))
            lbl = str(row['label']).strip()
            segs.append({'t0': float(ts[0]), 't1': float(ts[1]),
                        'is_laugh': 1 if lbl in ('B','I','L') else 0})
        seg_indices = np.where(mask)[0]
        # Predicted spans
        pred_spans, in_s, st = [], False, 0.0
        for k, si in enumerate(seg_indices):
            t0, t1 = segs[k]['t0'], segs[k]['t1']
            if vid_probs[k] >= pred_th and not in_s: in_s, st = True, t0
            elif vid_probs[k] < pred_th and in_s: in_s = False; pred_spans.append((st, segs[k-1]['t1']))
        if in_s: pred_spans.append((st, segs[-1]['t1']))
        # GT spans
        gt_spans, i = [], 0
        while i < len(segs):
            if segs[i]['is_laugh'] == 1:
                st, en = segs[i]['t0'], segs[i]['t1']
                j = i + 1
                while j < len(segs) and segs[j]['is_laugh'] == 1: en = segs[j]['t1']; j += 1
                gt_spans.append((st, en)); i = j
            else: i += 1
        for th in iou_thresholds:
            p, r, f = iou_f1(pred_spans, gt_spans, th)
            all_results[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})

print('\n=== IoU Results ===')
for th in iou_thresholds:
    rs = all_results[th]
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    fm = np.mean([x['f'] for x in rs])
    print(f'IoU>={th}: P={pm:.4f} R={rm:.4f} F1={fm:.4f}')

In [ ]:
# Save results
out = {'n_videos': n_vids, 'pred_th': pred_th,
       'iou_results': {th: {'macro_p': float(np.mean([x['p'] for x in all_results[th]])),
                            'macro_r': float(np.mean([x['r'] for x in all_results[th]])),
                            'macro_f1': float(np.mean([x['f'] for x in all_results[th]]))}
                   for th in iou_thresholds}}
with open(f'{BASE}/iou_xgboost_verified.json', 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved to {BASE}/iou_xgboost_verified.json')
print('\nPaper claim: F1=0.952 @ IoU=0.4')
print('Result:', out['iou_results'].get('0.4', {}))